# DataInsight — Text: ✏️ <project title>

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/trthminh/CO5177/blob/main/notebooks/text.ipynb)

| | |
|---|---|
| **Course** | Programming Foundations for Data Analysis and Visualization (CO5177) — Semester 261, 2026–2027 |
| **Lecturer** | Lê Thành Sách — Faculty of Computer Science and Engineering, HCMUT – VNU-HCM |
| **Group** | DataInsight — Trương Thanh Minh (2580893), Tô Anh Phát (2580894), Trần Lê Hoàng Long (2570443) |
| **Owner of this notebook** | Tô Anh Phát (2580894) |
| **Dataset** | ✏️ [<dataset name>](<link>) — <source>, <license> |
| **Task** | ✏️ Text classification (e.g. sentiment) of `<text column>` into `<k>` classes |

**Problem statement.** ✏️ One paragraph: what is predicted, from which inputs, why it matters, and which metric decides the best model.

**Table of contents**
1. [Setup](#s1)
2. [Data loading](#s2)
3. [Dataset description](#s3)
4. [Data quality checks](#s4)
5. [Exploratory data analysis (EDA)](#s5)
6. [Data preparation](#s6)
7. [Modeling](#s7)
8. [Evaluation on the test set](#s8)
9. [Inference on new data](#s9)
10. [Conclusion](#s10)

> **How to run.** Open in Google Colab → **Runtime → Run all**. The notebook installs missing libraries and downloads the data by itself; no manual upload, Drive mount or API key is needed.

> ### 🧭 How to use this template — delete this cell before submitting
> 1. Copy this file to `notebooks/` with its final name (the Colab badge above already points there) and keep the section numbers and headings unchanged, so the three notebooks of the group read the same way.
> 2. Fill in **section 1.3 (Configuration)** first. The notebook refuses to run while a `"TODO"` value is left there.
> 3. Run all once. Every generic cell (missing values, duplicates, split, pipelines, metrics) adapts to your columns by itself.
> 4. Then work through every `✏️ TODO` (markdown) and `# TODO` (code): dataset-specific rules, EDA questions, observations with **numbers from your own output**.
> 5. Cell pattern used everywhere: **markdown "what and why" → code with comments → markdown "Observation / Conclusion"**. Never leave a figure without a written conclusion.
> 6. Before submitting, go through the checklist in section 10 and `notebooks/templates/README.md`.
>
> Template source: `notebooks/templates/text_template.ipynb`.

<a id="s1"></a>
## 1. Setup

### 1.1 Install missing libraries

Google Colab already ships most of these libraries. The cell below only installs a package when Python cannot find it, so Run all stays fast on Colab and still works on a fresh local environment. `pyarrow` is listed because many text corpora (e.g. on Hugging Face) are distributed as Parquet files.

In [ ]:
import importlib.util
import subprocess
import sys

# import name -> pip requirement. Add a line here for every extra library you use.
REQUIRED_PACKAGES = {
    "numpy": "numpy",
    "pandas": "pandas>=2.0",
    "sklearn": "scikit-learn>=1.2",
    "plotly": "plotly>=5.15",
    "gdown": "gdown>=4.6",
    "kagglehub": "kagglehub",
    "pyarrow": "pyarrow",
}

# find_spec returns None when a module is not installed; nothing is imported yet, so no restart is needed.
missing = [spec for module, spec in REQUIRED_PACKAGES.items() if importlib.util.find_spec(module) is None]
if missing:
    print("Installing:", ", ".join(missing))
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
else:
    print("All required packages are already installed.")

### 1.2 Imports

In [ ]:
import itertools
import re
import time
import unicodedata                      # Unicode normalisation (NFC) — essential for Vietnamese diacritics
import urllib.request
import warnings
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots

import sklearn
from sklearn.base import clone
from sklearn.dummy import DummyClassifier                                   # baseline that ignores the text
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer  # text -> sparse matrix (N, V)
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, classification_report,
                             confusion_matrix, f1_score)
from sklearn.model_selection import GridSearchCV, StratifiedKFold, train_test_split
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline
from sklearn.svm import LinearSVC

warnings.filterwarnings("ignore", category=FutureWarning)
pd.set_option("display.max_columns", 50)
pd.set_option("display.max_colwidth", 120)   # show enough of each text in tables
pd.set_option("display.float_format", "{:,.3f}".format)

print(f"numpy {np.__version__} | pandas {pd.__version__} | scikit-learn {sklearn.__version__}")

### 1.3 Configuration

**Everything that depends on the chosen dataset is set here and only here.**

In [ ]:
# ---- Reproducibility --------------------------------------------------------------------------
SEED = 42
np.random.seed(SEED)

# ---- Data sources (section 2) -----------------------------------------------------------------
DATA_DIR = Path("data")
DATA_SOURCES = [  # one dict per file; several files (train/dev/test, one file per class, ...) are concatenated
    {"kind": "TODO", "location": "TODO", "filename": "TODO.csv"},
    # Add "split": "train" / "validation" / "test" to EVERY source to keep the dataset's official split.
]
READ_KWARGS = {}               # e.g. {"sep": "\t"} or {"encoding": "utf-8-sig"}

# ---- Problem definition -----------------------------------------------------------------------
TEXT_COL = "TODO"              # column holding the raw text
LABEL_COL = "TODO"             # column holding the class
LABEL_NAMES = {}               # optional readable names, e.g. {0: "negative", 1: "neutral", 2: "positive"}

# ---- Text processing --------------------------------------------------------------------------
TOKEN_PATTERN = r"(?u)\b\w+\b"  # a token = a run of letters/digits. sklearn's default needs 2+ characters and
                                # would silently drop 1-letter Vietnamese words such as "ở", "à", "ạ".
KEEP_EMOJI = False              # TODO: True if emojis carry the label (e.g. sentiment); decided in section 4.3
STOPWORDS = []                  # optional, used for the EDA word counts only (models keep every word)
SHORT_TEXT_WORDS = 3            # texts with fewer words are flagged in section 4.5

# ---- Split ------------------------------------------------------------------------------------
TEST_SIZE, VAL_SIZE = 0.15, 0.15   # used only when the dataset has no official split

# Stop Run all early with a clear message instead of failing 20 cells later.
for _name, _value in {"DATA_SOURCES": DATA_SOURCES, "TEXT_COL": TEXT_COL, "LABEL_COL": LABEL_COL}.items():
    assert "TODO" not in str(_value), f"Fill in {_name} in section 1.3 before running the notebook."

### 1.4 Plot style and shared helpers

All figures use **Plotly** (interactive: hover shows exact values, zoom works on long series). One shared template keeps colours and fonts identical across the three notebooks of the group — the palette is the same as in the Time series notebook.

Rules for every figure:
- Build it with `plotly.express` / `plotly.graph_objects`, then call `show_fig(fig, title)`. The helper numbers figures automatically (`Figure 1.`, `Figure 2.`, …), so numbering never breaks when a cell is added.
- The title states the **conclusion** ("Class 1 is only 12% of samples"), not the chart type ("Bar chart of classes").
- Label both axes with units, and follow every figure with a markdown observation that quotes numbers.

In [ ]:
# Colours by role, fixed order: categorical series take BLUE, ORANGE, AQUA, YELLOW, then RED/MUTED.
BLUE, ORANGE, AQUA, YELLOW = "#2a78d6", "#eb6834", "#1baf7a", "#eda100"
RED, MUTED, GRID = "#e34948", "#898781", "#e1e0d9"
QUALITATIVE = [BLUE, ORANGE, AQUA, YELLOW, RED, MUTED]
SEQ_BLUE = ["#cde2fb", "#9ec5f4", "#6da7ec", "#3987e5", "#256abf", "#184f95", "#0d366b"]  # counts, intensities
DIVERGING = [[0, BLUE], [0.5, "#f0efec"], [1, RED]]                                          # correlations (-1..1)

# A named Plotly template = one place to change the look of every chart.
pio.templates["datainsight"] = go.layout.Template(layout=dict(
    font=dict(family="Inter, Segoe UI, Arial, sans-serif", size=13, color="#0b0b0b"),
    colorway=QUALITATIVE,
    paper_bgcolor="#ffffff", plot_bgcolor="#ffffff",
    title=dict(x=0.01, font=dict(size=16)),
    xaxis=dict(gridcolor=GRID, linecolor="#c9c8c3", zeroline=False, title_font_color="#52514e"),
    yaxis=dict(gridcolor=GRID, linecolor="#c9c8c3", zeroline=False, title_font_color="#52514e"),
    legend=dict(orientation="h", yanchor="bottom", y=1.02, x=0, title_text=""),
    margin=dict(l=60, r=20, t=70, b=50),
))
pio.templates.default = "plotly_white+datainsight"

_figure_number = itertools.count(1)  # global counter: Run all numbers figures 1, 2, 3, ... in order


def show_fig(fig, title, height=420, **layout):
    """Give the figure its number and conclusion-style title, then display it."""
    fig.update_layout(title=f"Figure {next(_figure_number)}. {title}", height=height, **layout)
    fig.show()


def hist_bar(values, bins=40, color=BLUE, name=None):
    """Histogram pre-computed with NumPy: the notebook stores ~40 bars instead of every data point."""
    values = np.asarray(values, dtype=float)
    counts, edges = np.histogram(values[~np.isnan(values)], bins=bins)
    centers = (edges[:-1] + edges[1:]) / 2
    return go.Bar(x=centers, y=counts, width=np.diff(edges) * 0.92, marker_color=color, name=name,
                  hovertemplate="%{x:,.2f}: %{y:,}<extra></extra>")


def grid_position(i, ncols):
    """(row, col) in a make_subplots grid for the i-th panel, filled left to right."""
    return i // ncols + 1, i % ncols + 1

<a id="s2"></a>
## 2. Data loading

### 2.1 Download

The data is **never uploaded by hand**: the notebook fetches it from a public source and caches it in `data/` (ignored by git), so the lecturer only has to press Run all. `download_dataset` supports three kinds of source:

| `kind` | `location` | Library | Notes |
|---|---|---|---|
| `"url"` | direct download link | `urllib` (standard library) | GitHub raw, UCI, Hugging Face `resolve/...` links, … |
| `"gdrive"` | Google Drive **file id** (the part after `/d/` in the share link) | `gdown` | Share the file as *Anyone with the link → Viewer* |
| `"kaggle"` | dataset handle `owner/dataset-name` | `kagglehub` | Public datasets need no API key |

If the link points to a `.zip` archive, also set `"archive"`; the archive is extracted and `filename` is searched inside it.

In [ ]:
def download_dataset(source, data_dir=DATA_DIR):
    """Download one data source (skipped if already cached) and return the path of `source["filename"]`."""
    data_dir.mkdir(parents=True, exist_ok=True)
    kind, location, filename = source["kind"], source["location"], source["filename"]

    if kind == "kaggle":
        import kagglehub
        root = Path(kagglehub.dataset_download(location))  # kagglehub keeps its own cache
    else:
        root = data_dir
        archive = source.get("archive")
        target = data_dir / (archive or filename)
        if not target.exists():  # cache: never download twice
            if kind == "url":
                urllib.request.urlretrieve(location, target)
            elif kind == "gdrive":
                import gdown
                gdown.download(id=location, output=str(target), quiet=False)
            else:
                raise ValueError(f"Unknown source kind {kind!r}: use 'url', 'gdrive' or 'kaggle'.")
        if archive:
            with zipfile.ZipFile(target) as zf:
                zf.extractall(data_dir)

    matches = sorted(root.rglob(filename))  # rglob: the file may sit in a sub-folder of the archive
    if not matches:
        found = [p.relative_to(root).as_posix() for p in root.rglob("*") if p.is_file()][:20]
        raise FileNotFoundError(f"{filename!r} not found under {root}. Files found: {found}")
    print(f"Data file: {matches[0]} ({matches[0].stat().st_size / 1e6:,.1f} MB)")
    return matches[0]


def read_table(path, na_values=None, **kwargs):
    """Read a data file with the pandas reader that matches its extension."""
    suffix = path.suffix.lower()
    if suffix in {".csv", ".txt", ".data"}:
        return pd.read_csv(path, na_values=na_values, **kwargs)
    if suffix == ".tsv":
        return pd.read_csv(path, sep="\t", na_values=na_values, **kwargs)
    if suffix in {".xlsx", ".xls"}:
        return pd.read_excel(path, na_values=na_values, **kwargs)
    if suffix == ".parquet":
        return pd.read_parquet(path, **kwargs)
    if suffix in {".json", ".jsonl"}:
        return pd.read_json(path, lines=suffix == ".jsonl", **kwargs)
    raise ValueError(f"No reader configured for {suffix!r}; add one in read_table.")

### 2.2 Read and assemble the corpus

All files are concatenated into one table `corpus` with three columns: `text`, `label` and `official_split` (empty when the dataset has no official split). Working on fixed column names keeps the rest of the notebook dataset-independent.

In [ ]:
frames = []
for source in DATA_SOURCES:
    part = read_table(download_dataset(source), **READ_KWARGS)
    part.columns = part.columns.astype(str).str.strip()
    frames.append(part.assign(official_split=source.get("split")))
raw = pd.concat(frames, ignore_index=True)
assert TEXT_COL in raw.columns and LABEL_COL in raw.columns, f"Columns found: {list(raw.columns)}"

corpus = pd.DataFrame({
    "text": raw[TEXT_COL],
    "label": raw[LABEL_COL].map(LABEL_NAMES) if LABEL_NAMES else raw[LABEL_COL],
    "official_split": raw["official_split"],
})
assert corpus["label"].notna().sum() == raw[LABEL_COL].notna().sum(), "LABEL_NAMES does not cover every label value"
USE_OFFICIAL_SPLIT = corpus["official_split"].notna().all()

print(f"{len(corpus):,} texts from {len(DATA_SOURCES)} file(s); official split used: {USE_OFFICIAL_SPLIT}")
corpus.head()

### 2.3 Does the dataset meet the assignment requirements?

Assignment §4.3: at least 2,000 samples; Vietnamese data is encouraged; self-collected (crawled) data is valued.

In [ ]:
VIETNAMESE_CHARS = "àáạảãâầấậẩẫăằắặẳẵèéẹẻẽêềếệểễìíịỉĩòóọỏõôồốộổỗơờớợởỡùúụủũưừứựửữỳýỵỷỹđ"
has_vietnamese = corpus["text"].astype(str).str.lower().str.contains(f"[{VIETNAMESE_CHARS}]")

requirements = pd.DataFrame([
    ("At least 2,000 samples", f"{len(corpus):,} texts", len(corpus) >= 2000),
    ("At least 2 classes", f"{corpus['label'].nunique()} classes", corpus["label"].nunique() >= 2),
    ("Vietnamese (encouraged)", f"{has_vietnamese.mean():.1%} of texts contain Vietnamese letters", has_vietnamese.mean() > 0.5),
], columns=["Requirement (assignment §4.3)", "This dataset", "Met"])
requirements

<a id="s3"></a>
## 3. Dataset description

✏️ **TODO:** Origin (who collected the texts, from which platform, when), how labels were assigned (annotators, rules, star ratings…), licence, and — if self-collected — the crawling procedure.

| Label | Meaning | Example |
|---|---|---|
| ✏️ | … | … |

In [ ]:
corpus.groupby("label").head(3).sort_values("label")  # 3 raw examples per class: read them before any statistics

**Observation.** ✏️ What the texts look like: length, register (formal / social media), typos, abbreviations, emojis, teencode.

<a id="s4"></a>
## 4. Data quality checks

This section diagnoses; the rules are decided in 4.6 and applied in section 6.1.

### 4.1 Missing and empty texts

In [ ]:
text_str = corpus["text"].astype(str)
quality = pd.DataFrame({
    "check": ["text is missing (NaN)", "text is empty / whitespace only", "label is missing"],
    "count": [int(corpus["text"].isna().sum()),
              int((corpus["text"].notna() & text_str.str.strip().eq("")).sum()),
              int(corpus["label"].isna().sum())],
})
quality

### 4.2 Duplicates and conflicting labels

Duplicated texts are compared on a light key (Unicode NFC, lower case, collapsed spaces). Two issues:
- the same text in train and test lets the model "remember" the answer (**leakage**);
- the same text with **different labels** is label noise that no model can resolve.

In [ ]:
dup_key = text_str.str.normalize("NFC").str.lower().str.replace(r"\s+", " ", regex=True).str.strip()
labels_per_text = corpus.groupby(dup_key)["label"].nunique()

dup_table = pd.DataFrame({
    "check": ["exact duplicate texts", "duplicates after lower-case / spaces / NFC",
              "texts that appear with different labels"],
    "count": [int(text_str.duplicated().sum()), int(dup_key.duplicated().sum()), int((labels_per_text > 1).sum())],
})
display(dup_table)
conflicts = corpus[dup_key.isin(labels_per_text[labels_per_text > 1].index)].sort_values("text")
conflicts.head(10)

**Observation.** ✏️ Number of duplicates and conflicts, examples, decision (keep one copy; drop every copy of a conflicting text).

### 4.3 Noise audit

Each row counts the texts containing a pattern. The result decides the normalisation rules in 4.6 (e.g. keep emojis if they correlate with the label, mask URLs and numbers).

- **Not NFC**: Vietnamese letters can be stored *composed* (`ế` = 1 code point) or *decomposed* (`e` + 2 combining marks). They look identical but are different strings, so the same word would become two tokens. `unicodedata.normalize("NFC", …)` fixes it.

In [ ]:
PATTERNS = {
    "URL": r"https?://\S+|www\.\S+",
    "e-mail": r"[\w.+-]+@[\w-]+\.[\w.]+",
    "HTML tag": r"<[^>]+>",
    "emoji": "[\U0001F300-\U0001FAFF\U00002600-\U000027BF]",
    "digit": r"\d",
    "repeated character (3+), e.g. 'quáaaa'": r"(\w)\1{2,}",
}
# The back-reference \1 needs a capture group; pandas warns about groups in str.contains, which is harmless here.
warnings.filterwarnings("ignore", message="This pattern is interpreted as a regular expression")

has_shouting = text_str.map(lambda t: any(w.isupper() and len(w) >= 4 for w in t.split()))  # str.isupper is Unicode-aware
audit = pd.DataFrame(
    [(name, int(text_str.str.contains(pattern, regex=True).sum())) for name, pattern in PATTERNS.items()]
    + [("UPPER-CASE word (4+ letters)", int(has_shouting.sum())),
       ("not in Unicode NFC form", int((text_str != text_str.str.normalize("NFC")).sum()))],
    columns=["pattern", "texts containing it"])
audit["% of texts"] = audit["texts containing it"] / len(corpus) * 100
audit

In [ ]:
# Is a pattern more frequent in one class? (e.g. emojis in positive reviews) -> % of texts per class containing it
pattern_by_class = pd.DataFrame({name: text_str.str.contains(p, regex=True).groupby(corpus["label"]).mean() * 100
                                 for name, p in PATTERNS.items()})
pattern_by_class.round(1)

**Observation.** ✏️ Which patterns are frequent; whether any is class-specific (→ keep it as a token instead of deleting it); decision for `KEEP_EMOJI`.

### 4.4 Class balance

In [ ]:
counts = corpus["label"].value_counts()
shares = counts / counts.sum() * 100
fig = go.Figure(go.Bar(x=counts.index.astype(str), y=counts.values, marker_color=BLUE,
                       text=[f"{n:,} ({p:.1f}%)" for n, p in zip(counts, shares)], textposition="outside",
                       hovertemplate="%{x}: %{y:,} texts<extra></extra>"))
MINORITY = counts.idxmin()
show_fig(fig, f"Class balance: majority/minority ratio = {counts.max() / counts.min():.1f} (smallest class: '{MINORITY}')",
         xaxis_title="Label", yaxis_title="Texts")
print(f"Accuracy of always predicting the majority class: {shares.max():.1f}%")

**Observation.** ✏️ Shares and ratio → F1-macro as the main metric, `class_weight="balanced"` for the linear models.

### 4.5 Very short and very long texts

In [ ]:
corpus["n_chars"] = text_str.str.len()
corpus["n_words"] = text_str.str.split().str.len()
p99 = corpus["n_words"].quantile(0.99)
print(f"Texts with fewer than {SHORT_TEXT_WORDS} words: {(corpus['n_words'] < SHORT_TEXT_WORDS).sum():,}")
print(f"Texts longer than the 99th percentile ({p99:.0f} words): {(corpus['n_words'] > p99).sum():,}")
display(corpus.loc[corpus["n_words"] < SHORT_TEXT_WORDS, ["text", "label"]].head(5))
corpus.nlargest(3, "n_words")[["text", "label", "n_words"]]

**Observation.** ✏️ Are short texts still meaningful ("tốt", "ok")? Are very long ones real or concatenation errors?

### 4.6 Normalisation rules

Decided from 4.3 and applied to every text — training data and new data in section 9 — by **one** function:

1. Unicode **NFC** (composed form) so identical-looking words are identical strings.
2. **Lower case**: "Tốt" and "tốt" become one token.
3. HTML tags removed; URLs, e-mails and numbers replaced by placeholder words (`urltoken`, `emailtoken`, `numtoken`): the fact that a number is present may matter, its exact value rarely does.
4. Emojis kept as words (`emoji1f60d`) when `KEEP_EMOJI`, otherwise removed with the other punctuation.
5. Punctuation removed; `\w` is Unicode-aware, so letters with diacritics and the `_` used by word segmenters (`sản_phẩm`) are kept.
6. Repeated spaces collapsed.

> ✏️ Optional extension (creativity points if compared): Vietnamese **word segmentation** with `pyvi` (`ViTokenizer.tokenize`) or `underthesea` (`word_tokenize(text, format="text")`) joins syllables of one word (`sản phẩm` → `sản_phẩm`). Add it as step 7 and compare the scores with / without it in section 7.

In [ ]:
HTML_RE = re.compile(r"<[^>]+>")
URL_RE = re.compile(r"https?://\S+|www\.\S+")
EMAIL_RE = re.compile(r"[\w.+-]+@[\w-]+\.[\w.]+")
NUMBER_RE = re.compile(r"\d+(?:[.,]\d+)*")
EMOJI_RE = re.compile("[\U0001F300-\U0001FAFF\U00002600-\U000027BF]")
PUNCT_RE = re.compile(r"[^\w\s]")        # anything that is not a letter/digit/_ and not a space
SPACE_RE = re.compile(r"\s+")


def normalize_text(text):
    # Apply the normalisation rules of section 4.6 to one text.
    if pd.isna(text):
        return ""                            # str(NaN) would otherwise become the word "nan"
    text = unicodedata.normalize("NFC", str(text)).lower()
    text = HTML_RE.sub(" ", text)
    text = URL_RE.sub(" urltoken ", text)    # before e-mail/number rules: URLs contain digits and dots
    text = EMAIL_RE.sub(" emailtoken ", text)
    text = NUMBER_RE.sub(" numtoken ", text)
    if KEEP_EMOJI:
        text = EMOJI_RE.sub(lambda m: f" emoji{ord(m.group()):x} ", text)  # each emoji -> its own word
    text = PUNCT_RE.sub(" ", text)
    return SPACE_RE.sub(" ", text).strip()


corpus["clean"] = corpus["text"].map(normalize_text)
changed = corpus.loc[corpus["text"].astype(str) != corpus["clean"], ["text", "clean"]]
print(f"{len(changed):,} of {len(corpus):,} texts changed by normalisation")
changed.sample(min(5, len(changed)), random_state=SEED)   # before / after, side by side

<a id="s5"></a>
## 5. Exploratory data analysis (EDA)

EDA describes the whole corpus (on the normalised text). It does not fit anything used by the models: the model vectorizer is fitted on train only in section 6.

### 5.1 Are texts of different classes different in length?

In [ ]:
corpus["n_words"] = corpus["clean"].str.split().str.len()
classes = sorted(corpus["label"].unique(), key=str)
edges = np.histogram_bin_edges(corpus["n_words"], bins=40, range=(0, corpus["n_words"].quantile(0.99)))
fig = go.Figure()
for i, cls in enumerate(classes):
    counts_c, _ = np.histogram(corpus.loc[corpus["label"] == cls, "n_words"], bins=edges)
    fig.add_trace(go.Bar(x=(edges[:-1] + edges[1:]) / 2, y=counts_c / counts_c.sum() * 100, name=str(cls),
                         marker_color=QUALITATIVE[i % 6], opacity=0.65,
                         hovertemplate="%{x:.0f} words: %{y:.1f}%<extra>" + str(cls) + "</extra>"))
medians = corpus.groupby("label")["n_words"].median()
show_fig(fig, f"Length distribution per class (median words: {', '.join(f'{k}={v:.0f}' for k, v in medians.items())})",
         barmode="overlay", xaxis_title="Words per text (up to the 99th percentile)", yaxis_title="% of texts in the class")
corpus.groupby("label")["n_words"].describe()

**Observation.** ✏️ Is length informative? Quote the medians. Long-tail lengths matter for models with a maximum input length.

### 5.2 Which words and word pairs are most frequent?

`CountVectorizer` builds the vocabulary and counts every term in every text (a Bag of Words matrix of shape (N, V)). `ngram_range=(2, 2)` counts pairs of adjacent words, which in Vietnamese often recovers two-syllable words (`giảng viên`).

In [ ]:
def top_terms(texts, ngram_range, k=20):
    """k most frequent terms in `texts` for the given n-gram size."""
    cv = CountVectorizer(token_pattern=TOKEN_PATTERN, ngram_range=ngram_range, stop_words=STOPWORDS or None)
    counts = np.asarray(cv.fit_transform(texts).sum(axis=0)).ravel()   # total count of every term
    order = counts.argsort()[::-1][:k]
    return pd.Series(counts[order], index=cv.get_feature_names_out()[order])


fig = make_subplots(rows=1, cols=2, subplot_titles=("Top 20 words", "Top 20 word pairs (bigrams)"), horizontal_spacing=0.2)
for k, (ngram, color) in enumerate([((1, 1), BLUE), ((2, 2), ORANGE)], start=1):
    top = top_terms(corpus["clean"], ngram).iloc[::-1]
    fig.add_trace(go.Bar(x=top.values, y=top.index, orientation="h", marker_color=color, showlegend=False,
                         hovertemplate="%{y}: %{x:,}<extra></extra>"), row=1, col=k)
show_fig(fig, "Most frequent terms in the corpus", height=600)

**Observation.** ✏️ Domain words vs function words; does the stop-word list need adjusting?

### 5.3 Which words distinguish the classes?

For each class, the **log-odds ratio** compares how often a term appears inside the class vs outside it (add-one smoothing avoids division by zero):

$$\text{score}(t, c) = \log\frac{(n_{t,c} + 1) / (N_c + V)}{(n_{t,\bar c} + 1) / (N_{\bar c} + V)}$$

A high score means "typical of this class". Terms seen in fewer than 5 texts are ignored so rare typos do not dominate.

In [ ]:
cv = CountVectorizer(token_pattern=TOKEN_PATTERN, ngram_range=(1, 2), min_df=5)
bow = cv.fit_transform(corpus["clean"])
vocab = cv.get_feature_names_out()
V = len(vocab)

distinctive = {}
for cls in classes:
    in_class = (corpus["label"] == cls).to_numpy()
    n_in = np.asarray(bow[in_class].sum(axis=0)).ravel()
    n_out = np.asarray(bow[~in_class].sum(axis=0)).ravel()
    score = np.log((n_in + 1) / (n_in.sum() + V)) - np.log((n_out + 1) / (n_out.sum() + V))
    distinctive[cls] = pd.Series(score, index=vocab).nlargest(12)

ncols = min(3, len(classes))
nrows = -(-len(classes) // ncols)
fig = make_subplots(rows=nrows, cols=ncols, subplot_titles=[f"'{c}'" for c in classes], horizontal_spacing=0.18)
for i, cls in enumerate(classes):
    r, k = grid_position(i, ncols)
    s = distinctive[cls].iloc[::-1]
    fig.add_trace(go.Bar(x=s.values, y=s.index, orientation="h", marker_color=QUALITATIVE[i % 6], showlegend=False,
                         hovertemplate="%{y}: %{x:.2f}<extra></extra>"), row=r, col=k)
show_fig(fig, "Most class-specific terms (log-odds ratio vs the other classes)", height=420 * nrows)

**Observation.** ✏️ Are the distinctive terms sensible (e.g. "tốt", "hay" for positive)? Do bigrams such as "không tốt" appear — a sign that word order matters (unigram BoW cannot see negation)?

### 5.4 How large is the vocabulary, and how many words are rare?

A few frequent words cover most tokens (Zipf's law); many words appear once (typos, names). This guides `min_df` in the vectorizer.

In [ ]:
word_counts = top_terms(corpus["clean"], (1, 1), k=None)          # every word, sorted by frequency
coverage = word_counts.cumsum() / word_counts.sum() * 100
n_for_90 = int((coverage < 90).sum()) + 1
hapax = (word_counts == 1).mean() * 100

fig = go.Figure(go.Scatter(x=np.arange(1, len(coverage) + 1), y=coverage.values, mode="lines",
                           line=dict(color=BLUE, width=2), hovertemplate="top %{x:,} words: %{y:.1f}% of tokens<extra></extra>"))
fig.add_hline(y=90, line_dash="dot", line_color=MUTED)
fig.add_vline(x=n_for_90, line_dash="dot", line_color=MUTED)
show_fig(fig, f"{len(word_counts):,} distinct words; the top {n_for_90:,} cover 90% of tokens; {hapax:.0f}% appear only once",
         xaxis_title="Vocabulary rank (log scale)", yaxis_title="Cumulative % of tokens", xaxis_type="log")

**Observation.** ✏️ Vocabulary size, coverage, share of words seen once → choice of `min_df`.

### 5.5 Question 1: ✏️ <e.g. "Are longer texts more often negative?">

Pattern for every question: **question → figure with a conclusion title → numbers → conclusion**. Example: class shares across five length groups of equal size.

In [ ]:
length_group = pd.qcut(corpus["n_words"], q=5, duplicates="drop")
share = pd.crosstab(length_group, corpus["label"], normalize="index") * 100
fig = go.Figure([go.Bar(x=share.index.astype(str), y=share[c], name=str(c), marker_color=QUALITATIVE[i % 6],
                        hovertemplate="%{x} words: %{y:.1f}%<extra>" + str(c) + "</extra>")
                 for i, c in enumerate(share.columns)])
show_fig(fig, "✏️ <conclusion> — class shares by text length (5 equal-size groups)", barmode="stack",
         xaxis_title="Words per text", yaxis_title="% of texts in the group")
share.round(1)

**Conclusion.** ✏️

### 5.6 Question 2: ✏️ <question>

✏️ Ideas from the rubric: words/n-grams that separate classes, effect of negation, typical texts of a class, comparison with a self-collected sample.

In [ ]:
# TODO: analysis for question 2 (compute -> figure with show_fig -> print the numbers quoted in the conclusion).

**Conclusion.** ✏️

<a id="s6"></a>
## 6. Data preparation

### 6.1 Cleaning (before the split)

Rules decided in section 4, applied on the normalised text:
1. drop texts that are empty after normalisation (nothing left to learn from);
2. drop **every** copy of a text that appears with different labels (label noise);
3. keep one copy of duplicated texts — done **before** splitting, so a text can never be in both train and test.

In [ ]:
def clean_corpus(df):
    steps = [("raw", len(df))]
    out = df[df["clean"].str.len() > 0]
    steps.append(("empty after normalisation removed", len(out)))
    labels_per_text = out.groupby("clean")["label"].transform("nunique")
    out = out[labels_per_text == 1]
    steps.append(("conflicting labels removed", len(out)))
    out = out.drop_duplicates(subset="clean")     # keeps the first copy (train comes first if splits are official)
    steps.append(("duplicates removed", len(out)))
    return out.reset_index(drop=True), pd.DataFrame(steps, columns=["step", "texts"])


data, clean_steps = clean_corpus(corpus.dropna(subset=["text", "label"]))
clean_steps.assign(removed=lambda t: t["texts"].shift(1) - t["texts"])

### 6.2 Train / validation / test split

- If the dataset ships an **official split**, it is kept (results stay comparable with published work).
- Otherwise: 70 / 15 / 15 with `stratify` so the three sets keep the class proportions.
- Train fits the vectorizer and the models; validation selects the model; test is used **once**, in section 8.

In [ ]:
if USE_OFFICIAL_SPLIT:
    parts = {name: data[data["official_split"] == name] for name in ["train", "validation", "test"]}
    train_df, val_df, test_df = parts["train"], parts["validation"], parts["test"]
else:
    train_df, temp_df = train_test_split(data, test_size=TEST_SIZE + VAL_SIZE, stratify=data["label"],
                                         random_state=SEED)
    val_df, test_df = train_test_split(temp_df, test_size=TEST_SIZE / (TEST_SIZE + VAL_SIZE),
                                       stratify=temp_df["label"], random_state=SEED)

# Leakage check: no normalised text may appear in two sets.
assert not set(train_df["clean"]) & set(test_df["clean"]), "same text in train and test"
assert not set(train_df["clean"]) & set(val_df["clean"]), "same text in train and validation"

X_train, y_train = train_df["clean"], train_df["label"]
X_val, y_val = val_df["clean"], val_df["label"]
X_test, y_test = test_df["clean"], test_df["label"]

split_table = pd.concat({name: part["label"].value_counts(normalize=True) * 100
                         for name, part in [("train", train_df), ("validation", val_df), ("test", test_df)]}, axis=1)
split_table.loc["texts"] = [len(train_df), len(val_df), len(test_df)]
split_table.round(1)  # class % per set (+ number of texts): proportions must be similar

### 6.3 From text to numbers: TF-IDF, fitted on train only

`TfidfVectorizer` turns each text into a vector with one column per term of the vocabulary (shape **(N, V)**):

$$\text{tfidf}(t, d) = \text{tf}(t, d) \times \text{idf}(t), \qquad \text{idf}(t) = \ln\frac{1 + n}{1 + \text{df}(t)} + 1$$

`tf` counts the term in the text; `idf` is small for terms present in many texts, so very common words weigh less. Each row is then scaled to length 1.

- `fit` (train only) **fixes the vocabulary and the idf values**; `transform` reuses them. A word never seen in train has no column, so it is ignored in validation/test — exactly what happens with new data.
- `sublinear_tf=True` uses $1 + \ln(\text{tf})$: a word repeated 10 times is not 10 times more important.
- The matrix is **sparse**: almost every cell is 0, and only non-zero cells are stored.

In [ ]:
tfidf_check = TfidfVectorizer(token_pattern=TOKEN_PATTERN, ngram_range=(1, 2), min_df=2, sublinear_tf=True)
X_train_vec = tfidf_check.fit_transform(X_train)     # learns vocabulary + idf from train
X_test_vec = tfidf_check.transform(X_test)           # reuses them: same columns, no refit
density = X_train_vec.nnz / (X_train_vec.shape[0] * X_train_vec.shape[1])
print(f"train {X_train_vec.shape} | test {X_test_vec.shape} | non-zero cells: {density:.3%}")

# One text, its non-zero columns: what the model actually receives.
row = X_train_vec[0]
terms = tfidf_check.get_feature_names_out()[row.indices]
print("Text:", X_train.iloc[0])
pd.Series(row.data, index=terms, name="tf-idf weight").sort_values(ascending=False).head(10).to_frame()

<a id="s7"></a>
## 7. Modeling

### 7.1 Compare text representations and classifiers on validation

Each candidate is a `Pipeline(vectorizer → classifier)`, so the vectorizer is always fitted on the data the classifier is fitted on.

| Representation | Captures |
|---|---|
| word 1-gram | which words occur (no order) |
| word 1–2-gram | also adjacent pairs: `không tốt`, two-syllable words |
| char 2–5-gram (`char_wb`) | pieces of words inside word boundaries: robust to typos and missing diacritics |

| Classifier | Why |
|---|---|
| `DummyClassifier` | **baseline**: majority class |
| `MultinomialNB` | probabilistic model built for word counts; very fast |
| `LogisticRegression` | linear, outputs probabilities, interpretable weights per word |
| `LinearSVC` | linear, usually strongest on sparse TF-IDF |

`class_weight="balanced"` compensates class imbalance. Main metric: **F1-macro**.

In [ ]:
VECTORIZERS = {
    "word 1-gram": dict(analyzer="word", token_pattern=TOKEN_PATTERN, ngram_range=(1, 1), min_df=2, sublinear_tf=True),
    "word 1-2-gram": dict(analyzer="word", token_pattern=TOKEN_PATTERN, ngram_range=(1, 2), min_df=2, sublinear_tf=True),
    "char 2-5-gram": dict(analyzer="char_wb", ngram_range=(2, 5), min_df=2, sublinear_tf=True),
}
CLASSIFIERS = {
    "Multinomial NB": MultinomialNB(),
    "Logistic Regression": LogisticRegression(max_iter=2000, class_weight="balanced"),
    "Linear SVM": LinearSVC(class_weight="balanced"),
}


def evaluate(model, X, y):
    pred = model.predict(X)
    return {"accuracy": accuracy_score(y, pred), "balanced_accuracy": balanced_accuracy_score(y, pred),
            "f1_macro": f1_score(y, pred, average="macro")}


def make_model(vectorizer_params, classifier):
    return Pipeline([("vec", TfidfVectorizer(**vectorizer_params)), ("model", clone(classifier))])


fitted, rows = {}, []
baseline = make_model(VECTORIZERS["word 1-gram"], DummyClassifier(strategy="most_frequent")).fit(X_train, y_train)
fitted[("baseline", "majority class")] = baseline
rows.append({"representation": "baseline", "classifier": "majority class", **evaluate(baseline, X_val, y_val), "fit seconds": 0.0})
for (vec_name, vec_params), (clf_name, clf) in itertools.product(VECTORIZERS.items(), CLASSIFIERS.items()):
    start = time.perf_counter()
    model = make_model(vec_params, clf).fit(X_train, y_train)
    seconds = time.perf_counter() - start
    fitted[(vec_name, clf_name)] = model
    rows.append({"representation": vec_name, "classifier": clf_name, **evaluate(model, X_val, y_val), "fit seconds": seconds})
    print(f"done: {vec_name} + {clf_name} ({seconds:.1f}s)")

results = pd.DataFrame(rows).set_index(["representation", "classifier"]).sort_values("f1_macro", ascending=False)
results

In [ ]:
grid = results.drop(index="baseline", level=0)["f1_macro"].unstack("classifier")
fig = go.Figure(go.Heatmap(z=grid.values, x=grid.columns, y=grid.index, colorscale=SEQ_BLUE,
                           text=grid.round(3).values, texttemplate="%{text}", colorbar=dict(title="F1-macro"),
                           hovertemplate="%{y} + %{x}: %{z:.3f}<extra></extra>"))
best_vec, best_clf = results.drop(index="baseline", level=0)["f1_macro"].idxmax()
show_fig(fig, f"Validation F1-macro: best = {best_vec} + {best_clf} "
              f"({grid.loc[best_vec, best_clf]:.3f} vs baseline {results.loc[('baseline', 'majority class'), 'f1_macro']:.3f})",
         xaxis_title="Classifier", yaxis_title="Representation", height=400)

**Observation.** ✏️ Best representation and classifier; does the 1–2-gram help (negation, compound words)? Does char n-gram help with typos? Gap to the baseline.

### 7.2 Hyperparameter tuning

Grid search with stratified 3-fold cross-validation **on train** around the best combination: regularisation strength (`C` for linear models, `alpha` for Naive Bayes) and `min_df`.

In [ ]:
PARAM_GRIDS = {  # TODO: widen or narrow the grids if needed
    "Multinomial NB": {"model__alpha": [0.1, 0.3, 1.0]},
    "Logistic Regression": {"model__C": [0.3, 1, 3, 10]},
    "Linear SVM": {"model__C": [0.1, 0.3, 1, 3]},
}
param_grid = {**PARAM_GRIDS[best_clf], "vec__min_df": [1, 2, 5]}

search = GridSearchCV(make_model(VECTORIZERS[best_vec], CLASSIFIERS[best_clf]), param_grid, scoring="f1_macro",
                      cv=StratifiedKFold(n_splits=3, shuffle=True, random_state=SEED), n_jobs=-1)
search.fit(X_train, y_train)
fitted[(best_vec, f"{best_clf} (tuned)")] = search.best_estimator_
tuned_row = pd.DataFrame([{"representation": best_vec, "classifier": f"{best_clf} (tuned)",
                           **evaluate(search.best_estimator_, X_val, y_val), "fit seconds": np.nan}])
results = pd.concat([results, tuned_row.set_index(["representation", "classifier"])])
print("Best parameters:", search.best_params_, f"| CV F1-macro = {search.best_score_:.3f}")
(pd.DataFrame(search.cv_results_)[["params", "mean_test_score", "std_test_score", "rank_test_score"]]
   .sort_values("rank_test_score").head(10))

### 7.3 Choose the final model

Best validation F1-macro; refitted on train + validation (hyperparameters frozen) before the single evaluation on test.

In [ ]:
results = results.sort_values("f1_macro", ascending=False)
BEST_KEY = results.drop(index="baseline", level=0).index[0]
final_model = clone(fitted[BEST_KEY]).fit(pd.concat([X_train, X_val]), pd.concat([y_train, y_val]))
print(f"Final model: {' + '.join(BEST_KEY)} (validation F1-macro {results.loc[BEST_KEY, 'f1_macro']:.3f})")
results

<a id="s8"></a>
## 8. Evaluation on the test set

### 8.1 Test metrics

In [ ]:
test_scores = pd.DataFrame({"Baseline (majority class)": evaluate(baseline, X_test, y_test),
                            f"Final: {' + '.join(BEST_KEY)}": evaluate(final_model, X_test, y_test)}).T
y_pred = final_model.predict(X_test)
CLASS_NAMES = [str(c) for c in final_model.classes_]
display(test_scores)
pd.DataFrame(classification_report(y_test, y_pred, output_dict=True)).T

### 8.2 Confusion matrix

In [ ]:
cm = confusion_matrix(y_test, y_pred, labels=final_model.classes_)
cm_row = cm / cm.sum(axis=1, keepdims=True) * 100
fig = go.Figure(go.Heatmap(z=cm_row, x=CLASS_NAMES, y=CLASS_NAMES, colorscale=SEQ_BLUE, zmin=0, zmax=100,
                           text=[[f"{n:,}<br>({p:.1f}%)" for n, p in zip(rn, rp)] for rn, rp in zip(cm, cm_row)],
                           texttemplate="%{text}", colorbar=dict(title="row %"),
                           hovertemplate="true %{y}, predicted %{x}: %{z:.1f}%<extra></extra>"))
worst = CLASS_NAMES[int(np.argmin(np.diag(cm_row)))]
show_fig(fig, f"Confusion matrix on test — lowest recall: '{worst}' ({np.diag(cm_row).min():.1f}%)",
         xaxis_title="Predicted class", yaxis_title="True class", yaxis_autorange="reversed", height=460)

**Observation.** ✏️ Which classes are confused, with numbers.

### 8.3 Misclassified examples

The rubric asks for the confusion matrix **with** examples of wrong predictions. ✏️ Read them and name the causes (negation, sarcasm, mixed opinions, label noise, typos).

In [ ]:
errors = test_df.assign(predicted=y_pred)
errors = errors[errors["label"] != errors["predicted"]]
print(f"{len(errors):,} wrong predictions out of {len(test_df):,} ({len(errors) / len(test_df):.1%})")
errors[["text", "label", "predicted"]].sample(min(15, len(errors)), random_state=SEED)

**Observation.** ✏️

### 8.4 Which terms drive the predictions?

For linear models, each class has one weight per term: the largest weights are the terms that push a text towards that class. Naive Bayes uses the log-probability of each term per class instead.

In [ ]:
vec, clf = final_model.named_steps["vec"], final_model.named_steps["model"]
terms = vec.get_feature_names_out()
if hasattr(clf, "coef_"):
    coef = clf.coef_ if clf.coef_.shape[0] > 1 else np.vstack([-clf.coef_[0], clf.coef_[0]])  # binary: one row only
elif hasattr(clf, "feature_log_prob_"):
    coef = clf.feature_log_prob_ - clf.feature_log_prob_.mean(axis=0)   # relative to the average class
else:
    coef = None

if coef is not None:
    ncols = min(3, len(CLASS_NAMES))
    nrows = -(-len(CLASS_NAMES) // ncols)
    fig = make_subplots(rows=nrows, cols=ncols, subplot_titles=[f"'{c}'" for c in CLASS_NAMES], horizontal_spacing=0.18)
    for i, cls in enumerate(CLASS_NAMES):
        top = pd.Series(coef[i], index=terms).nlargest(12).iloc[::-1]
        r, k = grid_position(i, ncols)
        fig.add_trace(go.Bar(x=top.values, y=top.index, orientation="h", marker_color=QUALITATIVE[i % 6],
                             showlegend=False, hovertemplate="%{y}: %{x:.3f}<extra></extra>"), row=r, col=k)
    show_fig(fig, "Terms with the largest weight per class in the final model", height=420 * nrows)

**Observation.** ✏️ Do the model's terms agree with the EDA in 5.3?

<a id="s9"></a>
## 9. Inference on new data

New raw texts → the **same** `normalize_text` → final pipeline (vectorizer fitted on train + validation → classifier).

In [ ]:
NEW_TEXTS = [  # TODO: write 3-5 realistic texts, including a hard one (negation, mixed opinion)
    "TODO: first example text",
    "TODO: second example text",
]
new = pd.DataFrame({"text": NEW_TEXTS})
new["normalized"] = new["text"].map(normalize_text)
new["predicted"] = final_model.predict(new["normalized"])
if hasattr(final_model, "predict_proba"):           # LinearSVC has no probabilities
    new["confidence"] = final_model.predict_proba(new["normalized"]).max(axis=1)
new

<a id="s10"></a>
## 10. Conclusion

### 10.1 Summary of results

✏️ Fill this table with the numbers printed in sections 7–8; the same numbers must appear in the PDF report and on the project page (`assets/js/data.js → results`).

| Representation + model | Accuracy | Balanced accuracy | F1-macro |
|---|---|---|---|
| Baseline (majority class) | ✏️ | ✏️ | ✏️ |
| ✏️ final model | ✏️ | ✏️ | ✏️ |

### 10.2 What the analysis answered

✏️ 3–5 numbered findings, each with a number from this notebook and a link to its figure (e.g. "Figure 7 shows …").

### 10.3 Limitations

✏️ Data (size, period, sampling bias, label noise), method (assumptions, features not used), evaluation (single split, test period/distribution).

### 10.4 Extensions

✏️ Only list what you would actually try next. Ideas that were run and compared earn creativity points; ideas that were not run do not.

### 10.5 Submission checklist — delete before submitting

- [ ] **Runtime → Restart and run all** on Google Colab finishes without errors, and the saved notebook keeps its outputs.
- [ ] No `TODO`, `✏️` or placeholder text is left (search the notebook for `TODO`).
- [ ] Every figure is Plotly, has a numbered conclusion-style title and labelled axes, and is followed by an observation that quotes numbers.
- [ ] Every number written in markdown matches the printed output of the current run.
- [ ] Imputers, scalers, encoders and vectorizers are fitted on **train only**; the test set is used once, in section 8.
- [ ] The model comparison includes a baseline, and the chosen model is selected on **validation**, not test.
- [ ] Duplicates and conflicting labels are removed before the split; the leakage asserts in 6.2 pass.
- [ ] Section 8.3 shows misclassified examples with an explanation of the error causes.
- [ ] `NEW_TEXTS` in section 9 contains real example sentences, not placeholders.
- [ ] `assets/js/data.js` is updated: notebook path, dataset, problem, results.